In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.layers import Dense,LSTM, Dropout
from keras.models import Sequential
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from matplotlib.pyplot import rcParams
rcParams['figure.figsize'] = 15, 6

In [4]:
data = pd.read_csv('/content/AMZN.csv')
data.head(10)

,Date,Open,High,Low,Close,Adj Close,Volume
0,1997-05-15,0.121875,0.125000,0.096354,0.097917,0.097917,1443120000
1,1997-05-16,0.098438,0.098958,0.085417,0.086458,0.086458,294000000
2,1997-05-19,0.088021,0.088542,0.081250,0.085417,0.085417,122136000
3,1997-05-20,0.086458,0.087500,0.081771,0.081771,0.081771,109344000
4,1997-05-21,0.081771,0.082292,0.068750,0.071354,0.071354,377064000
5,1997-05-22,0.071875,0.072396,0.065625,0.069792,0.069792,235536000
6,1997-05-23,0.070313,0.076042,0.066667,0.075000,0.075000,318744000
7,1997-05-27,0.075521,0.082292,0.072917,0.079167,0.079167,173952000
8,1997-05-28,0.081250,0.081771,0.076563,0.076563,0.076563,91488000
9,1997-05-29,0.077083,0.077083,0.073958,0.075260,0.075260,69456000


In [15]:
data.shape

(6684, 7)

In [13]:
treino = data.iloc[:5347, 1:2].values

In [14]:
scaler = MinMaxScaler()

treino = scaler.fit_transform(treino)

In [16]:
treino.shape

(5347, 1)

In [17]:
x_train = []
y_train = []
timestamp = 180

In [18]:
for i in range(timestamp, len(treino)):
  x_train.append(treino[i - timestamp:i, 0])
  y_train.append(treino[i, 0])

In [19]:
x_train = np.array(x_train)
y_train = np.array(y_train)

In [20]:
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

In [21]:
model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(Dropout(0.2))

model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(256, return_sequences=False))
model.add(Dropout(0.2))

model.add(Dense(1))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 180, 64)           16896     
                                                                 
 dropout (Dropout)           (None, 180, 64)           0         
                                                                 
 lstm_1 (LSTM)               (None, 180, 128)          98816     
                                                                 
 dropout_1 (Dropout)         (None, 180, 128)          0         
                                                                 
 lstm_2 (LSTM)               (None, 256)               394240    
                                                                 
 dropout_2 (Dropout)         (None, 256)               0         
                                                                 
 dense (Dense)               (None, 1)                 2

In [ ]:
model.compile(optimizer='adam', loss='mean_squared_error', metrics='mean_squared_error')
model.fit(x_train, y_train, epochs = 20, batch_size = 32)

In [ ]:
y_test = data.iloc[5347:, 1:2].values
y_test = scaler.fit_transform(y_test)

In [ ]:
x_test = []

length = len(y_test)

In [ ]:
for i in range(timestamp, length):
    x_test.append(y_test[i - timestamp:i, 0])

In [ ]:
x_test = np.array(x_test)
x_test = x_test.reshape(x_test.shape[0], x_test.shape[1], 1)

In [ ]:
x_train.shape, x_test.shape

((5257, 90, 1), (1247, 90, 1))

In [ ]:
y_pred = model.predict(x_test)
y_pred = scaler.inverse_transform(y_pred)
escala_min, escala_max = min(y_test), max(y_test)
y_pred = (y_pred * (escala_max - escala_min)) + escala_min

39/39 [==============================] - 6s 155ms/step


In [ ]:
y_test = scaler.inverse_transform(y_test)
y_test

array([[ 95.969498],
       [ 95.477501],
       [ 95.196999],
       ...,
       [146.      ],
       [145.25    ],
       [143.550003]])

In [ ]:
y_pred.shape, y_test.shape

((1247, 1), (1337, 1))

In [ ]:
y_test.mean() - y_pred.mean()

-1.5825191234616653

In [ ]:
plt.plot(y_test, color='b', label='Preço real das ações')
plt.plot(y_pred, color='r', label='Previsão das ações')

plt.title('Previsão do Preço das Ações')
plt.xlabel('Tempo')
plt.ylabel('Preço das Ações')
plt.legend()

plt.show()